In [0]:
import os
import json
import base64
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
from os import urandom

# 1. KEY SETUP (Databricks Secrets nundi teesukuntundi)
try:
    # Databricks environment lo unte idi work avtundi
    RAW_KEY = dbutils.secrets.get('courseify', 'data-encription-key')
except NameError:
    # Local ga test chestunte, ikkada 32-character key ivvu
    RAW_KEY = "your-32-char-very-secret-key-123" 

# Key ni exactly 32 bytes (AES-256) ki align chestunnam
SECRET_KEY = RAW_KEY.encode('utf-8').ljust(32, b'\0')[:32]

def embed_images(data, base_dir):
    """JSON lo 'note_image_path' unte vatini Base64 loki marchi embed chestundi."""
    if isinstance(data, dict):
        if "note_image_path" in data:
            path = data["note_image_path"]
            
            # Full path lekapothe, JSON file unna folder lo vethukutundi
            if not os.path.isabs(path):
                path = os.path.join(base_dir, path)
                
            if os.path.exists(path):
                with open(path, "rb") as img_f:
                    encoded = base64.b64encode(img_f.read()).decode("utf-8")
                    data["note_image_base64"] = encoded
                print(f"  🖼️  Embedded Image: {os.path.basename(path)}")
            else:
                print(f"  ⚠️  Warning: Image not found at {path}")
            
            # Path ni delete chesi, base64 matrame unchutunnam
            del data["note_image_path"]
        
        for key, value in data.items():
            embed_images(value, base_dir)
    elif isinstance(data, list):
        for item in data:
            embed_images(item, base_dir)

def process_and_encrypt(input_file_path):
    """JSON ni read chesi, images embed chesi, .enc file generate chestundi."""
    
    # Path setup
    if not os.path.exists(input_file_path):
        print(f"❌ Error: File '{input_file_path}' dorkaledu. Path check chesko.")
        return

    base_name = os.path.splitext(input_file_path)[0]
    output_file = f"{base_name}.enc"
    base_dir = os.path.dirname(os.path.abspath(input_file_path))
    
    try:
        print(f"📂 Processing File: {input_file_path}")
        
        # 1. Load JSON Data
        with open(input_file_path, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
            
        # 2. Embed Images
        print("🔄 Embedding images into JSON...")
        embed_images(json_data, base_dir)
        
        # 3. String to Bytes
        data_str = json.dumps(json_data)
        data_bytes = data_str.encode('utf-8')
        
        # 4. PKCS7 Padding (AES blocks are 16 bytes)
        padding_len = 16 - (len(data_bytes) % 16)
        padded_data = data_bytes + bytes([padding_len] * padding_len)
        
        # 5. AES CBC Encryption
        iv = urandom(16)
        cipher = Cipher(algorithms.AES(SECRET_KEY), modes.CBC(iv), backend=default_backend())
        encryptor = cipher.encryptor()
        encrypted_payload = encryptor.update(padded_data) + encryptor.finalize()
        
        # 6. Save Result (IV + Encrypted Data)
        with open(output_file, 'wb') as f:
            f.write(iv + encrypted_payload)
            
        print("-" * 30)
        print(f"✅ SUCCESS!")
        print(f"📄 Output File: {output_file}")
        print("-" * 30)
        
    except Exception as e:
        print(f"❌ Unexpected Error: {str(e)}")

MY_JSON_PATH = "/Workspace/Users/courseify.help@gmail.com/courseify-backend-data/json/courses_index.json" 

# Execution
process_and_encrypt(MY_JSON_PATH)

In [0]:
%sql
select * from list_secrets()

In [0]:
import sys
import os
import json
import base64
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
from os import urandom

# 1. KEY SETUP
# Databricks lo unte idi unchu, lekapothe direct string ivvu
try:
    RAW_KEY = dbutils.secrets.get('courseify', 'data-encription-key')
except NameError:
    # Nuvvu local ga test chestunte ikada nee key ivvu
    RAW_KEY = "your_secret_key_here" 

SECRET_KEY = RAW_KEY.encode('utf-8').ljust(32, b'\0')[:32]

def embed_images(data, base_dir):
    """Recursively search for 'note_image_path' and embed it as Base64."""
    if isinstance(data, dict):
        if "note_image_path" in data:
            path = data["note_image_path"]
            # Full path lekapothe JSON unna folder lo vethukutundi
            if not os.path.isabs(path):
                path = os.path.join(base_dir, path)
                
            if os.path.exists(path):
                with open(path, "rb") as img_f:
                    encoded = base64.b64encode(img_f.read()).decode("utf-8")
                    data["note_image_base64"] = encoded
                print(f"  🖼️ Embedded: {os.path.basename(path)}")
            else:
                print(f"  ⚠️ Warning: Image not found at {path}")
            
            del data["note_image_path"]
        
        for key, value in data.items():
            embed_images(value, base_dir)
    elif isinstance(data, list):
        for item in data:
            embed_images(item, base_dir)

def encrypt_data(input_file_path):
    # Extension marchi output file name set chestundi (.json -> .enc)
    base_name = os.path.splitext(input_file_path)[0]
    output_file = f"{base_name}.enc"
    base_dir = os.path.dirname(os.path.abspath(input_file_path))
    
    try:
        print(f"📂 Processing: {input_file_path}")
        
        # 1. Load JSON
        with open(input_file_path, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
            
        # 2. Embed Images (Base64)
        embed_images(json_data, base_dir)
        
        # 3. Convert to Bytes
        data_bytes = json.dumps(json_data).encode('utf-8')
        
        # 4. PKCS7 Padding
        padding_length = 16 - (len(data_bytes) % 16)
        padded_data = data_bytes + bytes([padding_length] * padding_length)
        
        # 5. AES CBC Encryption
        iv = urandom(16)
        cipher = Cipher(algorithms.AES(SECRET_KEY), modes.CBC(iv), backend=default_backend())
        encryptor = cipher.encryptor()
        encrypted_payload = encryptor.update(padded_data) + encryptor.finalize()
        
        # 6. Save (IV + Encrypted Data)
        with open(output_file, 'wb') as f:
            f.write(iv + encrypted_payload)
            
        print(f"✅ Success! Encrypted file created: {output_file}")
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")

if __name__ == "__main__":
    # Terminal nundi path pass cheyali: python script.py mydata.json
    if len(sys.argv) < 2:
        path = input("Enter the JSON file path: ").strip('"')
        encrypt_data(path)
    else:
        encrypt_data(sys.argv[1])